# Diagnostic: Events Before Price History

Goal:
- Identify stocks where dividend events exist *before* the first available price date.
- Count how many such stocks exist.
- Compare their per-stock AUC-PR with the rest.


In [41]:
import polars as pl
import glob

files = glob.glob(r"D:/Users/Workspace/WQ-Project/code/data/raw/crsp_sp500_2000_2024_download/*.parquet")

lfs = [pl.scan_parquet(f) for f in files]

lf = pl.concat(lfs, how="vertical_relaxed")  # 关键：先用 relaxed 合并，避免 schema mismatch

lf = lf.with_columns(
    pl.col("DlyNumTrd")
      .cast(pl.Utf8)
      .str.replace_all(",", "")          # 如果有千分位
      .cast(pl.Int64, strict=False)      # 转不了就变 null，不报错
      .alias("DlyNumTrd")
)

# 不想占内存就直接 sink（推荐）
lf.sink_parquet("tableB.parquet")

# 或者你也可以：df = lf.collect(); df.write_parquet("tableB.parquet")


In [4]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

In [11]:
df = pd.read_csv('D:/Users/Workspace/WQ-Project/code/data/raw/tableB.csv')

C:\Users\19672\AppData\Local\Temp\ipykernel_20724\3437688494.py:1: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('D:/Users/Workspace/WQ-Project/code/data/raw/tableB.csv')


In [15]:
df2 = pd.read_csv("C:/Users/19672/Desktop/tableB.csv")

C:\Users\19672\AppData\Local\Temp\ipykernel_20724\3282884488.py:1: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("C:/Users/19672/Desktop/tableB.csv")


In [16]:
df2.columns

Index(['INDNO', 'MbrStartDt', 'MbrEndDt', 'PERMNO', 'HdrCUSIP', 'PERMCO',
       'SICCD', 'CUSIP', 'Ticker', 'ShareType', 'SecurityType',
       'SecuritySubType', 'USIncFlg', 'IssuerType', 'PrimaryExch',
       'ConditionalType', 'TradingStatusFlg', 'DlyCalDt', 'DlyPrc',
       'DlyPrcFlg', 'DlyCap', 'DlyCapFlg', 'DlyRet', 'DlyRetx',
       'DlyRetMissFlg', 'DlyRetDurFlg', 'DlyOrdDivAmt', 'DlyNonOrdDivAmt',
       'DlyFacPrc', 'DlyDistRetFlg', 'DlyVol', 'DlyClose', 'DlyLow', 'DlyHigh',
       'DlyBid', 'DlyAsk', 'DlyOpen', 'DlyNumTrd', 'DlyMMCnt', 'DlyCumFacPr',
       'DlyCumFacShr', 'ShrOut'],
      dtype='object')

In [ ]:
df.columns

Index(['INDNO', 'MbrStartDt', 'MbrEndDt', 'PERMNO', 'HdrCUSIP', 'PERMCO',
       'SICCD', 'CUSIP', 'Ticker', 'ShareType', 'SecurityType',
       'SecuritySubType', 'USIncFlg', 'IssuerType', 'PrimaryExch',
       'ConditionalType', 'TradingStatusFlg', 'DlyCalDt', 'DlyPrc',
       'DlyPrcFlg', 'DlyCap', 'DlyCapFlg', 'DlyRet', 'DlyRetx',
       'DlyRetMissFlg', 'DlyRetDurFlg', 'DlyOrdDivAmt', 'DlyNonOrdDivAmt',
       'DlyFacPrc', 'DlyDistRetFlg', 'DlyVol', 'DlyClose', 'DlyLow', 'DlyHigh',
       'DlyBid', 'DlyAsk', 'DlyOpen', 'DlyNumTrd', 'DlyMMCnt', 'DlyCumFacPr',
       'DlyCumFacShr', 'ShrOut'],
      dtype='object')

In [18]:
DATE_COL = "DlyCalDt"

In [19]:
if ("MbrStartDt" in df.columns) and ("MbrEndDt" in df.columns):
    df["in_sp500"] = ((df["MbrStartDt"] <= df[DATE_COL]) & (df[DATE_COL] <= df["MbrEndDt"])).astype("int8")

In [22]:
df['in_sp500'].sum() == df.__len__()

True

In [14]:
df[df['PERMNO'] == 32942]

,INDNO,MbrStartDt,MbrEndDt,PERMNO,HdrCUSIP,PERMCO,SICCD,CUSIP,Ticker,ShareType,...,DlyLow,DlyHigh,DlyBid,DlyAsk,DlyOpen,DlyNumTrd,DlyMMCnt,DlyCumFacPr,DlyCumFacShr,ShrOut
3003151,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,287.565,298.630,289.88,289.93,298.63,NaN,NaN,1.0,1.0,53631.0
3003654,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,283.980,295.120,284.98,284.99,290.68,NaN,NaN,1.0,1.0,53631.0
3004157,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,277.760,284.190,279.87,279.88,284.01,NaN,NaN,1.0,1.0,53631.0
3004660,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,279.665,283.795,280.37,280.38,280.38,NaN,NaN,1.0,1.0,53631.0
3005163,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,280.920,284.815,282.08,282.26,283.52,NaN,NaN,1.0,1.0,53631.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3153047,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,426.300,433.130,432.04,432.61,426.30,NaN,NaN,1.0,1.0,53671.0
3153550,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,429.230,433.790,430.46,430.61,430.60,NaN,NaN,1.0,1.0,53671.0
3154053,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,421.670,430.090,422.79,422.98,426.78,NaN,NaN,1.0,1.0,53671.0
3154556,1000500,2023-10-18,2024-12-31,32942,44351060,20945,3699,44351060,HUBB,NS,...,412.690,421.220,418.85,418.94,417.79,NaN,NaN,1.0,1.0,53671.0


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.utils.paths import load_yaml, resolve_paths
from src.eval.eval_tools import per_stock_metrics, validate_eval_df

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


ModuleNotFoundError: No module named 'code.src'; 'code' is not a package

In [ ]:
# Parameters (edit if needed)
RUN_ID = '2026-01-27_193843'
SPLIT = 'test'
PATHS_CFG = 'configs/paths.yaml'
CONFIG_CFG = 'configs/config.yaml'

paths = resolve_paths(load_yaml(Path(PATHS_CFG)), project_root=ROOT)
cfg = load_yaml(Path(CONFIG_CFG))

DIV_DISTCD = set(cfg['div_distcd'])
H_LABEL = int(cfg.get('H_label', 10))

run_dir = paths.outputs_dir / RUN_ID
preds_dir = run_dir / 'preds'
eval_dir = run_dir / 'eval'

run_dir, preds_dir.exists(), eval_dir.exists()


In [ ]:
# Load eval_df (prefers preds/eval_df.parquet, then <split>_preds.parquet)
def _read_parquet_or_csv(parquet_path: Path) -> pd.DataFrame:
    csv_path = parquet_path.with_suffix('.csv')
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception:
            if csv_path.exists():
                return pd.read_csv(csv_path)
            raise
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Missing {parquet_path} and {csv_path}')

def load_eval_df(preds_dir: Path, split: str) -> pd.DataFrame:
    candidates = [preds_dir / 'eval_df.parquet', preds_dir / f'{split}_preds.parquet']
    last_err = None
    for p in candidates:
        try:
            df = _read_parquet_or_csv(p)
            return validate_eval_df(df)
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f'Could not load eval_df from {candidates}. Last error: {last_err}')

eval_df = load_eval_df(preds_dir, SPLIT)
eval_df['date'] = pd.to_datetime(eval_df['date'])
eval_min, eval_max = eval_df['date'].min(), eval_df['date'].max()

eval_df.shape, eval_min, eval_max


In [ ]:
# Per-stock span within eval_df
stock_span = (
    eval_df.groupby('permno')['date']
    .agg(stock_min='min', stock_max='max', n_rows='size')
    .reset_index()
)
stock_span.head()


In [ ]:
# Compute first available price date per stock from raw tableB (chunked)
def compute_price_start(table_b_path: Path, chunksize: int = 400_000) -> pd.DataFrame:
    min_dates: dict[int, pd.Timestamp] = {}
    for chunk in pd.read_csv(table_b_path, usecols=['PERMNO', 'DlyCalDt'], chunksize=chunksize, low_memory=True):
        chunk = chunk.dropna(subset=['PERMNO', 'DlyCalDt'])
        if chunk.empty:
            continue
        chunk['PERMNO'] = chunk['PERMNO'].astype(int)
        d = pd.to_datetime(chunk['DlyCalDt'], errors='coerce')
        chunk = chunk.assign(_date=d).dropna(subset=['_date'])
        grp = chunk.groupby('PERMNO')['_date'].min()
        for permno, dt in grp.items():
            p = int(permno)
            prev = min_dates.get(p)
            min_dates[p] = dt if prev is None else min(prev, dt)
    out = pd.DataFrame({'permno': list(min_dates.keys()), 'price_start': list(min_dates.values())})
    out['price_start'] = pd.to_datetime(out['price_start'])
    return out

price_start = compute_price_start(paths.tableB_path)
price_start.head(), len(price_start)


In [ ]:
# Load dividend events from raw tableA
events = pd.read_csv(paths.tableA_path, usecols=['PERMNO', 'DCLRDT', 'DISTCD'])
events['DCLRDT'] = pd.to_datetime(events['DCLRDT'], errors='coerce')
events = events.dropna(subset=['PERMNO', 'DCLRDT'])
events['PERMNO'] = events['PERMNO'].astype(int)
events = events[events['DISTCD'].isin(DIV_DISTCD)].copy()
events = events.rename(columns={'PERMNO': 'permno', 'DCLRDT': 'event_date'})
events['event_date'] = pd.to_datetime(events['event_date'])

events_total = (
    events.groupby('permno')['event_date']
    .agg(first_event='min', last_event='max', n_events_total='size')
    .reset_index()
)
events_total.head()


In [ ]:
# Build diagnostic table and count events before price start / in eval span
diag = (
    stock_span
    .merge(price_start, on='permno', how='left')
    .merge(events_total, on='permno', how='left')
)

events_by_perm = {p: g['event_date'].values.astype('datetime64[ns]') for p, g in events.groupby('permno', sort=False)}

def count_events(arr, start=None, end=None) -> int:
    if arr is None or len(arr) == 0:
        return 0
    a = arr
    if start is not None:
        a = a[a >= np.datetime64(start)]
    if end is not None:
        a = a[a <= np.datetime64(end)]
    return int(len(a))

before_price = []
in_stock_span = []
in_eval_range = []

for r in diag.itertuples(index=False):
    arr = events_by_perm.get(int(r.permno))
    ps = r.price_start
    before_price.append(count_events(arr, end=(ps - pd.Timedelta(days=1))) if pd.notna(ps) else 0)
    in_stock_span.append(count_events(arr, start=r.stock_min, end=r.stock_max))
    in_eval_range.append(count_events(arr, start=eval_min, end=eval_max))

diag['n_events_before_price_start'] = before_price
diag['n_events_in_stock_span'] = in_stock_span
diag['n_events_in_eval_range'] = in_eval_range
diag['late_price_start'] = diag['n_events_before_price_start'] > 0

diag[['late_price_start', 'n_events_before_price_start']].groupby('late_price_start').describe()


In [ ]:
# Attach per-stock AUC/AUC-PR metrics from eval_df
stock_metrics = per_stock_metrics(eval_df)
diag = diag.merge(stock_metrics, on='permno', how='left')

# Optionally attach full-history cohort stats for context
coh_full_path = eval_dir / 'stock_cohorts_full.csv'
if coh_full_path.exists():
    coh_full = pd.read_csv(coh_full_path)
    keep = [c for c in ['permno', 'gap_cv', 'gap_med', 'n_events'] if c in coh_full.columns]
    coh_full = coh_full[keep].rename(columns={
        'gap_cv': 'gap_cv_full_hist',
        'gap_med': 'gap_med_full_hist',
        'n_events': 'n_events_full_hist',
    })
    diag = diag.merge(coh_full, on='permno', how='left')

diag.head()


In [ ]:
# Unit-test style checks (sanity assertions)
assert (diag['late_price_start'] == (diag['n_events_before_price_start'] > 0)).all()

mask = diag['late_price_start'] & diag['first_event'].notna() & diag['price_start'].notna()
assert (diag.loc[mask, 'first_event'] < diag.loc[mask, 'price_start']).all()

late_count = int(diag['late_price_start'].sum())
late_count, len(diag)


In [ ]:
# Examples: stocks with the most events before price start
late_examples = (
    diag[diag['late_price_start']]
    .sort_values(['n_events_before_price_start', 'n_events_total'], ascending=False)
    .head(20)
)
late_examples[[
    'permno', 'price_start', 'first_event', 'n_events_before_price_start',
    'n_events_in_stock_span', 'n_events_in_eval_range', 'n_pos', 'aucpr'
]].head(10)


In [ ]:
# Compare AUC-PR between late-price-start stocks and others
diag_npos = diag[diag['n_pos'] > 0].copy()

def auc_stats(g: pd.DataFrame) -> pd.Series:
    s = pd.to_numeric(g['aucpr'], errors='coerce').dropna()
    if len(s) == 0:
        return pd.Series({'n_stocks': 0})
    return pd.Series({
        'n_stocks': int(len(s)),
        'mean_aucpr': float(s.mean()),
        'median_aucpr': float(s.median()),
        'p25_aucpr': float(s.quantile(0.25)),
        'p75_aucpr': float(s.quantile(0.75)),
        'min_aucpr': float(s.min()),
        'max_aucpr': float(s.max()),
    })

auc_by_group = diag_npos.groupby('late_price_start', dropna=False).apply(auc_stats).reset_index()
auc_by_group


In [ ]:
# Simple effect size: difference in mean AUC-PR
if set(auc_by_group['late_price_start']) >= {False, True}:
    mean_late = float(auc_by_group.loc[auc_by_group['late_price_start'] == True, 'mean_aucpr'].iloc[0])
    mean_nonlate = float(auc_by_group.loc[auc_by_group['late_price_start'] == False, 'mean_aucpr'].iloc[0])
    delta_mean = mean_late - mean_nonlate
else:
    mean_late = mean_nonlate = delta_mean = np.nan

mean_late, mean_nonlate, delta_mean


In [ ]:
# Bin by number of events before price start and compare AUC-PR
bins = [-1, 0, 5, 20, 50, 100, 1_000_000]
labels = ['0', '1-5', '6-20', '21-50', '51-100', '100+']
diag_npos['before_price_bin'] = pd.cut(diag_npos['n_events_before_price_start'], bins=bins, labels=labels)

bin_stats = (
    diag_npos.groupby('before_price_bin', dropna=False)['aucpr']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .reset_index()
)
bin_stats


In [ ]:
# Inspect late-price-start stocks with the lowest AUC-PR
late_low_auc = (
    diag_npos[diag_npos['late_price_start']]
    .sort_values('aucpr', ascending=True)
    .head(20)
)
late_low_auc[[
    'permno', 'aucpr', 'auc', 'n', 'n_pos', 'pos_rate',
    'price_start', 'first_event', 'n_events_before_price_start',
    'n_events_in_stock_span', 'n_events_in_eval_range',
    'gap_cv_full_hist', 'gap_med_full_hist', 'n_events_full_hist'
]].head(10)
